# ProtCell cell-type annotation

Load a frozen ProtCell backbone, train a classification head, test held-out cells or batches, and transfer cell labels to another dataset. Measured markers and the protein volcano plot help inspect transferred annotations.

Run the cells in order with the `protein_llm` kernel. Edit the data and checkpoint paths in the setup cell. The `RUN_*` switches enable the corresponding fine-tuning, prediction or analysis steps.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path(os.environ.get("PROTCELL_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
if not (PROJECT_ROOT / "train.py").is_file() and (PROJECT_ROOT.parent / "train.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train.py").is_file(), "Set PROTCELL_PROJECT_ROOT to your ProtCell clone before starting JupyterLab."
OUTPUT_ROOT = Path(os.environ.get("PROTCELL_OUTPUT_DIR", str(PROJECT_ROOT / "results/outputs")))
OUTPUT_DIR = OUTPUT_ROOT / "annotation"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESULT_DIR = OUTPUT_DIR / "results"
MODEL_DIR = OUTPUT_DIR / "models"
for directory in [OUTPUT_DIR, FIGURE_DIR, RESULT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.chdir(OUTPUT_DIR) # trainer logs and new files stay with this experiment

old_cache_settings = {key: os.environ.get(key) for key in ["MPLCONFIGDIR", "NUMBA_CACHE_DIR", "XDG_CACHE_HOME"]}
plot_cache = tempfile.TemporaryDirectory(prefix="protcell-annotation-", dir=OUTPUT_DIR)
os.environ["MPLCONFIGDIR"] = plot_cache.name
os.environ["NUMBA_CACHE_DIR"] = str(Path(plot_cache.name) / "numba")
os.environ["XDG_CACHE_HOME"] = str(Path(plot_cache.name) / "xdg")

import json
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse, stats
from tqdm.auto import tqdm
from IPython.display import display
from model.model import ProteinLanguageModel
from trainer.mask_trainer import ProteinSequenceDataset, collate_fn
from torch.utils.data import DataLoader
from finetuner.Annotation import DistributionMLP, AnnotationTrainer
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "sans-serif"})
sc.settings.figdir = FIGURE_DIR
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_ROOT = Path(os.environ.get("PROTCELL_DATA_ROOT", str(PROJECT_ROOT / "data"))).expanduser().resolve()
BASE_MODEL_PATH = PROJECT_ROOT / "checkpoints/basemodel/large-scale-new-10/model_369999.pt"
print("Device:", DEVICE)
print("Outputs:", OUTPUT_DIR)

## Load Kotliarov2020 and define batches

Split units are explicit. Five-fold annotation uses cells from batch 1; cross-batch tests use the other batch.

In [ ]:
adata = ad.read_h5ad(DATA_ROOT / "CLR_data/Dataset83.h5ad")
batch1 = adata[adata.obs["batch"].astype(str).eq("1")].copy()
batch2 = adata[adata.obs["batch"].astype(str).eq("2")].copy()
assert not set(batch1.obs_names).intersection(batch2.obs_names)
label_key = "celltype"
RUN_FIVE_FOLD_TRAINING = True
RUN_CROSS_BATCH_TRAINING = True
RUN_CROSS_BATCH_TEST = True
RUN_CROSS_DATASET_TEST = True
RUN_MARKER_ANALYSIS = True
display(batch1)
display(batch2)

## Load the pretrained backbone

Load pretrained ProtCell and freeze its backbone. Train classification heads below, or set RUN_CROSS_BATCH_TRAINING=False to load saved heads from head_paths.


In [ ]:
model_path = BASE_MODEL_PATH
assert BASE_MODEL_PATH.is_file(), f"Pretrained ProtCell checkpoint not found: {BASE_MODEL_PATH}"
model = ProteinLanguageModel.load_model_from_checkpoint(str(model_path), device=DEVICE).to(DEVICE)
model.requires_grad_(False)
model.eval()
head_paths = {
    "1": PROJECT_ROOT / "checkpoints/annotation/inter_batch/83_batch1/head_20.pt",
    "2": PROJECT_ROOT / "checkpoints/annotation/inter_batch/83_batch2/head_20.pt",
}
trained_heads = {}
display(model)

## Basic prediction function

Define a batched prediction function for cell-type annotation.

In [ ]:
def predict_cell_type(model, 
                        adata, 
                        dist_head,
                        device='cuda',
                        batch_size=2048,
                        predict_mapping=None,
                        show_tqdm = True):
    tokenizer = model.tokenizer

    model.eval()
    model.to(device)
    dist_head.eval()
    dist_head.to(device)

    celltype2id = dist_head.celltype2id
    id2celltype = {v:k for k,v in celltype2id.items()}

    dataset = ProteinSequenceDataset(adata, tokenizer)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    celltype_pred_ids = []
    a = tqdm(enumerate(dataloader), total=len(dataloader), desc="Predict Protein Values") if show_tqdm else enumerate(dataloader)
    with torch.no_grad():
        for _, batch in a:
            input_ids = batch["input_ids"].to(device)          # (B, L)
            expr_values = batch["expr_values"].to(device)      # (B, L)

            cell_embed = model(
                input_ids=input_ids, 
                expr_values=expr_values,
                mask=None,
                return_cls=True
            ) # (B, D)
            logits = dist_head(cell_embed)
            preds = torch.argmax(logits, dim=-1) # (B)

            # 4. 存入列表 (转回 CPU 以释放显存)
            celltype_pred_ids.extend(preds.cpu().numpy())

    pred_celltype = [id2celltype[idx] for idx in celltype_pred_ids]
    if predict_mapping is not None:
        pred_celltype = [predict_mapping.get(ct, ct) for ct in pred_celltype]
    adata.obs['predicted_celltype'] = pred_celltype

    return pred_celltype

## Five-fold training and held-out testing

Train a new classification head on each training fold using the frozen pretrained backbone. Keep the freshly calculated fold metrics in memory for the summary below; export copies to CSV.


In [ ]:
within_batch_metrics = None
if RUN_FIVE_FOLD_TRAINING:
    fold_rows = []
    splitter = KFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (train_idx, test_idx) in enumerate(splitter.split(np.arange(batch1.n_obs)), 1):
        train_data = batch1[train_idx].copy()
        test_data = batch1[test_idx].copy()
        assert not set(train_data.obs_names).intersection(test_data.obs_names)
        model = ProteinLanguageModel.load_model_from_checkpoint(str(BASE_MODEL_PATH), device=DEVICE).to(DEVICE)
        model.requires_grad_(False)
        annotation_trainer = AnnotationTrainer(
            model=model, adata=train_data, celltype_name=label_key, trainable_modules={},
            lr=1e-3, batch_size=512, device=DEVICE, expr_name=f"within_batch/fold{fold}", seed=SEED,
        )
        annotation_trainer.fit(num_epochs=20, save=True, save_dir=str(MODEL_DIR),
                               save_epoch_interval=20, early_stop=False)
        prediction = predict_cell_type(model, test_data, annotation_trainer.dist_head,
                                       device=DEVICE, batch_size=512)
        truth = test_data.obs[label_key].astype(str)
        fold_dir = RESULT_DIR / f"within_batch/fold{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(classification_report(truth, prediction, output_dict=True, zero_division=0)).T.to_csv(fold_dir / "classification_report.csv")
        pd.DataFrame({"cell_id": test_data.obs_names, "true_celltype": truth.to_numpy(),
                      "predicted_celltype": prediction}).to_csv(fold_dir / "predictions.csv", index=False)
        pd.DataFrame({"cell_id": list(train_data.obs_names) + list(test_data.obs_names),
                      "split": ["train"] * train_data.n_obs + ["test"] * test_data.n_obs}).to_csv(fold_dir / "split.csv", index=False)
        fold_rows.append({"fold": fold, "Accuracy": accuracy_score(truth, prediction),
                          "Macro F1": f1_score(truth, prediction, average="macro"),
                          "Weighted F1": f1_score(truth, prediction, average="weighted")})
    within_batch_metrics = pd.DataFrame(fold_rows)
    within_batch_metrics.to_csv(RESULT_DIR / "new_within_batch_metrics.csv", index=False)

## Train cross-batch heads

Each head trains on its source batch only. The pretrained backbone is reused and remains frozen.

In [ ]:
trained_heads = {}
if RUN_CROSS_BATCH_TRAINING:
    for training_batch, training_data in [("1", batch1), ("2", batch2)]:
        model = ProteinLanguageModel.load_model_from_checkpoint(str(BASE_MODEL_PATH), device=DEVICE).to(DEVICE)
        model.requires_grad_(False)
        annotation_trainer = AnnotationTrainer(
            model=model, adata=training_data, celltype_name=label_key, trainable_modules={},
            lr=1e-3, batch_size=512, device=DEVICE, expr_name=f"batch{training_batch}", seed=SEED,
        )
        annotation_trainer.fit(num_epochs=20, save=True, save_dir=str(MODEL_DIR),
                               save_epoch_interval=20, early_stop=False)
        trained_heads[training_batch] = annotation_trainer.dist_head

## Test both cross-batch directions

The confusion matrix is normalized by the reference label. Its rows correspond to true cell types.

In [ ]:
cross_batch_metrics = None
if RUN_CROSS_BATCH_TEST:
    model = ProteinLanguageModel.load_model_from_checkpoint(str(BASE_MODEL_PATH), device=DEVICE).to(DEVICE)
    cross_rows = []
    for training_batch, test_data in [("1", batch2), ("2", batch1)]:
        test_data = test_data.copy()
        if RUN_CROSS_BATCH_TRAINING:
            assert training_batch in trained_heads, "Run the cross-batch head training cell first."
            head = trained_heads[training_batch]
        else:
            assert head_paths[training_batch].is_file(), f"Annotation checkpoint not found:: {head_paths[training_batch]}"
            head = DistributionMLP.load_head(str(head_paths[training_batch]), device=DEVICE)
        prediction = predict_cell_type(model, test_data, head, device=DEVICE, batch_size=512)
        truth = test_data.obs[label_key].astype(str).to_numpy()
        direction = f"batch{training_batch}_to_batch{3 - int(training_batch)}"
        pd.DataFrame(classification_report(truth, prediction, output_dict=True, zero_division=0)).T.to_csv(RESULT_DIR / f"{direction}_report.csv")
        pd.DataFrame({"cell_id": test_data.obs_names, "true_celltype": truth,
                      "predicted_celltype": prediction}).to_csv(RESULT_DIR / f"{direction}_predictions.csv", index=False)
        labels = sorted(set(truth).union(prediction))
        cm = confusion_matrix(truth, prediction, labels=labels, normalize="true")
        fig, ax = plt.subplots(figsize=(8, 7))
        sns.heatmap(pd.DataFrame(cm, index=labels, columns=labels), cmap="Blues", ax=ax)
        ax.set_xlabel("Predicted label"); ax.set_ylabel("Reference label")
        fig.savefig(FIGURE_DIR / f"{direction}_confusion_matrix.pdf", bbox_inches="tight"); plt.show()
        cross_rows.append(dict(direction=direction, Accuracy=accuracy_score(truth, prediction),
                               Macro_F1=f1_score(truth, prediction, average="macro")))
    cross_batch_metrics = pd.DataFrame(cross_rows)
    cross_batch_metrics.to_csv(RESULT_DIR / "new_cross_batch_metrics.csv", index=False)

## Transfer the frozen model to Sacco2022

Apply the classifier trained on Kotliarov2020 batch 1 to Sacco2022. Visualize reference and transferred labels on a shared UMAP computed from the measured protein profiles.

In [ ]:
transfer_data = ad.read_h5ad(DATA_ROOT / "CLR_data/Dataset74.h5ad")
if RUN_CROSS_DATASET_TEST:
    model = ProteinLanguageModel.load_model_from_checkpoint(str(BASE_MODEL_PATH), device=DEVICE).to(DEVICE)
    if RUN_CROSS_BATCH_TRAINING:
        assert "1" in trained_heads, "Run the cross-batch head training cell first."
        head = trained_heads["1"]
    else:
        assert head_paths["1"].is_file(), f"Annotation checkpoint not found:: {head_paths['1']}"
        head = DistributionMLP.load_head(str(head_paths["1"]), device=DEVICE)
    prediction = predict_cell_type(model, transfer_data, head, device=DEVICE, batch_size=512)
    transfer_data.obs["predicted_celltype"] = pd.Categorical(prediction)
    sc.pp.pca(transfer_data, n_comps=50, random_state=SEED)
    sc.pp.neighbors(transfer_data, use_rep="X_pca")
    sc.tl.umap(transfer_data, random_state=SEED)
    transfer_data.write_h5ad(RESULT_DIR / "Sacco_transferred_labels.h5ad", compression="gzip")
    sc.pl.umap(transfer_data, color=["coarsecelltype", "predicted_celltype"], show=False)
    plt.savefig(FIGURE_DIR / "Sacco_annotation_transfer.pdf", bbox_inches="tight"); plt.show()

## Measured T-cell markers

Calculate mean expression and the fraction above each marker’s median. The dotplot uses measured values only.

In [ ]:
if RUN_MARKER_ANALYSIS:
    assert RUN_CROSS_DATASET_TEST and "predicted_celltype" in transfer_data.obs, "Run annotation transfer before marker analysis."
    celltypes = ["CD4+ Naive T", "CD4+ Memory T", "CD8+ Naive T (some DNT)", "CD8+ Memory T"]
    markers = [p for p in ["CD4", "CD8A", "CD45RA", "SELL", "CD45RO", "CD5", "CD28", "FAS", "KLRK1", "ITGAL", "KLRG1"]
               if p in transfer_data.var_names]
    subset = transfer_data[transfer_data.obs.predicted_celltype.isin(celltypes), markers].copy()
    X = subset.X.toarray() if sparse.issparse(subset.X) else np.asarray(subset.X)
    thresholds = np.nanmedian(X, axis=0)
    rows = []
    for celltype in celltypes:
        group = X[subset.obs.predicted_celltype.astype(str).eq(celltype).to_numpy()]
        if len(group) == 0:
            continue
        for j, protein in enumerate(markers):
            rows.append(dict(celltype=celltype, protein=protein, mean=np.nanmean(group[:, j]),
                             fraction=np.nanmean(group[:, j] > thresholds[j])))
    marker_summary = pd.DataFrame(rows)
    marker_summary.to_csv(RESULT_DIR / "transferred_Tcell_markers.csv", index=False)
    fig, ax = plt.subplots(figsize=(9, 4))
    scatter = ax.scatter(marker_summary.protein, marker_summary.celltype, s=15 + 180 * marker_summary.fraction,
                         c=marker_summary["mean"], cmap="viridis")
    fig.colorbar(scatter, ax=ax, label="Mean measured expression")
    ax.tick_params(axis="x", rotation=45); fig.tight_layout()
    fig.savefig(FIGURE_DIR / "transferred_Tcell_dotplot.pdf", bbox_inches="tight"); plt.show()

## Naive/memory protein contrasts

Run the actual differential-analysis calls directly.

In [ ]:
try:
    from adjustText import adjust_text
    HAS_ADJUST_TEXT = True
except ImportError:
    HAS_ADJUST_TEXT = False


def plot_protein_volcano(
    adata,
    group,
    reference,
    groupby="predicted_celltype",
    method="wilcoxon",
    pval_cutoff=0.05,
    logfc_cutoff=0.5,
    top_n_label=10,
    figsize=(10, 8),
    title=None,
    save_path=None
):
    """
    Plot volcano plot for pairwise differential protein analysis.

    group: target cell type, e.g. "T_Vd2"
    reference: reference cell type, e.g. "CD8"
    """

    sc.tl.rank_genes_groups(
        adata,
        groupby=groupby,
        groups=[group],
        reference=reference,
        method=method,
        use_raw=False
    )

    diff_df = sc.get.rank_genes_groups_df(adata, group=group).copy()

    diff_df["names"] = diff_df["names"].astype(str)
    diff_df["logfoldchanges"] = pd.to_numeric(diff_df["logfoldchanges"], errors="coerce")
    diff_df["pvals_adj"] = pd.to_numeric(diff_df["pvals_adj"], errors="coerce")

    # 避免 adjusted p-value = 0 时 -log10 变成无限大
    positive_pvals = diff_df.loc[diff_df["pvals_adj"] > 0, "pvals_adj"]
    min_positive = positive_pvals.min() if len(positive_pvals) > 0 else 1e-300
    diff_df["pvals_adj_plot"] = diff_df["pvals_adj"].replace(0, min_positive * 0.1)

    diff_df["minus_log10_padj"] = -np.log10(diff_df["pvals_adj_plot"])

    diff_df["status"] = "NS"
    diff_df.loc[
        (diff_df["pvals_adj"] < pval_cutoff) &
        (diff_df["logfoldchanges"] > logfc_cutoff),
        "status"
    ] = f"Up in {group}"

    diff_df.loc[
        (diff_df["pvals_adj"] < pval_cutoff) &
        (diff_df["logfoldchanges"] < -logfc_cutoff),
        "status"
    ] = f"Up in {reference}"

    color_map = {
        "NS": "#C9CED6",
        f"Up in {group}": "#00A6FF",       # 明亮蓝
        f"Up in {reference}": "#FF4D6D"    # 明亮红
    }

    fig, ax = plt.subplots(figsize=figsize)

    for status, color in color_map.items():
        adata_sub = diff_df[diff_df["status"] == status]

        ax.scatter(
            adata_sub["logfoldchanges"],
            adata_sub["minus_log10_padj"],
            s=80 if status != "NS" else 45,
            c=color,
            label=status,
            alpha=0.9 if status != "NS" else 0.45,
            edgecolors="white",
            linewidths=0.6
        )

    # 阈值线
    ax.axvline(logfc_cutoff, color="#555555", linestyle="--", linewidth=1.3)
    ax.axvline(-logfc_cutoff, color="#555555", linestyle="--", linewidth=1.3)
    ax.axhline(-np.log10(pval_cutoff), color="#555555", linestyle="--", linewidth=1.3)

    # 选择要标注的蛋白
    label_df = diff_df[
        (diff_df["status"] != "NS")
    ].copy()

    label_df["abs_logfoldchanges"] = label_df["logfoldchanges"].abs()

    label_df = label_df.sort_values(
        ["abs_logfoldchanges", "minus_log10_padj"],
        ascending=[False, False]
    ).head(top_n_label)

    texts = []

    for _, row in label_df.iterrows():
        text = ax.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=14,
            fontweight="bold",
            color="#1F2937"
        )
        texts.append(text)

    if HAS_ADJUST_TEXT and len(texts) > 0:
        adjust_text(
            texts,
            ax=ax,
            arrowprops=dict(
                arrowstyle="-",
                color="#777777",
                lw=0.8
            ),
            expand_points=(1.2, 1.4),
            expand_text=(1.2, 1.4),
            force_text=(0.5, 0.8),
            force_points=(0.3, 0.5)
        )

    if title is None:
        title = f"{group} vs {reference}"

    ax.set_title(title, fontsize=22, fontweight="bold", pad=16)
    ax.set_xlabel("log2 fold change", fontsize=18)
    ax.set_ylabel("-log10 adjusted p-value", fontsize=18)

    ax.tick_params(axis="both", labelsize=15)

    ax.legend(
        frameon=True,
        fontsize=14,
        loc="lower right",
        borderpad=0.8
    )

    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, format="pdf", bbox_inches="tight")
        plt.savefig(save_path.replace(".pdf", ".png"), dpi=300, bbox_inches="tight")

    plt.show()

    return diff_df

In [ ]:
if RUN_MARKER_ANALYSIS:
    for group, reference in [("CD4+ Memory T", "CD4+ Naive T"),
                              ("CD8+ Memory T", "CD8+ Naive T (some DNT)")]:
        counts = transfer_data.obs.predicted_celltype.value_counts()
        if counts.get(group, 0) < 2 or counts.get(reference, 0) < 2:
            print("Skipping contrast: fewer than two cells in either group.", group, reference)
            continue
        differential = plot_protein_volcano(
            transfer_data,
            group=group,
            reference=reference,
            groupby="predicted_celltype",
            method="wilcoxon",
            pval_cutoff=0.05,
            logfc_cutoff=0.5,
            top_n_label=10,
            figsize=(10, 8),
            save_path=str(FIGURE_DIR / f"{group}_vs_{reference}.pdf"),
        )
        differential.to_csv(RESULT_DIR / f"{group}_vs_{reference}.csv", index=False)


## Summarize ProtCell annotation performance

Display the fold-level and cross-batch metrics calculated in this run. Disabled experiments contribute no rows.


In [ ]:
annotation_tables = []
for scenario, enabled, metrics in [
    ("Within batch", RUN_FIVE_FOLD_TRAINING, within_batch_metrics),
    ("Cross batch", RUN_CROSS_BATCH_TEST, cross_batch_metrics),
]:
    if enabled:
        assert metrics is not None, f"Run the {scenario} experiment before summarizing it."
        frame = metrics.copy().rename(columns={"Macro_F1": "Macro F1", "Weighted_F1": "Weighted F1"})
        frame["scenario"] = scenario
        annotation_tables.append(frame)
if annotation_tables:
    annotation_metrics = pd.concat(annotation_tables, ignore_index=True)
    display(annotation_metrics)
else:
    print("Run the within-batch or cross-batch test cells to save annotation metrics.")

## Release temporary files

Keep the final tables, figures and trained models; delete this notebook’s temporary plotting/JIT caches.

In [ ]:
plt.close("all")
plot_cache.cleanup()
for key, previous in old_cache_settings.items():
    if previous is None:
        os.environ.pop(key, None)
    else:
        os.environ[key] = previous
print("Temporary caches removed.")